# 07_adipose_label_transfer

Selected source cells for review and reconstruction.

Source: `Adipose_combined_samples_CellSegmentation/Adipose_Combination_4(label transfer_snRNAseq_Tabula to All).ipynb`

Load adipose_mergeobj_preProject_sketchHarmony.fixed_onDiskPath_joinCounts.rds as merge.obj first. Disk-path repair trials excluded.

Original execution counts and plots were removed. Do not run in the source-data folder: original output filenames and Docker paths are retained as provenance. Use the common scripts for new runs. These excerpts have not been executed as a fresh notebook.


In [ ]:
merge.obj <- readRDS("adipose_mergeobj_preProject_sketchHarmony.fixed_onDiskPath_joinCounts.rds")


Source cell 1 (one-based); original execution count 1.

In [ ]:
library(Seurat)
library(BPCells)
library(Matrix)

Source cell 28 (one-based); original execution count 31.

In [ ]:
# scRNA-seq reference
ref_obj <- readRDS("adipose_merged_obj_reference.rds")

Source cell 29 (one-based); original execution count 32.

In [ ]:
# 最小確認
ref_obj
DefaultAssay(ref_obj) <- "RNA"
head(colnames(ref_obj[[]]))
head(rownames(ref_obj), 5)

Source cell 31 (one-based); original execution count 35.

In [ ]:
DefaultAssay(ref_obj)  <- "RNA"
DefaultAssay(merge.obj) <- "RNA"

genes_use <- intersect(rownames(ref_obj), rownames(merge.obj))
length(genes_use)

Source cell 32 (one-based); original execution count 36.

In [ ]:
library(future)
plan(sequential)


Source cell 33 (one-based); original execution count 38.

In [ ]:
DefaultAssay(ref_obj)  <- "RNA"
DefaultAssay(merge.obj) <- "RNA"

ref_obj   <- FindVariableFeatures(ref_obj,  verbose = FALSE)
merge.obj <- FindVariableFeatures(merge.obj, verbose = FALSE)

features_use <- intersect(VariableFeatures(ref_obj), VariableFeatures(merge.obj))
length(features_use)

# 必要なら上限（265なら実質そのまま）
features_use <- head(features_use, 500)


Source cell 34 (one-based); original execution count 39.

In [ ]:
# anchors
anchors <- FindTransferAnchors(
  reference = ref_obj,
  query = merge.obj,
  features = features_use,
  dims = 1:30
)

# transfer
pred <- TransferData(
  anchorset = anchors,
  refdata = ref_obj$Cellname,
  dims = 1:30
)

# attach
merge.obj <- AddMetaData(merge.obj, metadata = pred)

Source cell 36 (one-based); original execution count 42.

In [ ]:
merge.obj$Cellname_final <- ifelse(
  merge.obj$prediction.score.max < 0.4,
  "Unknown",
  merge.obj$predicted.id
)

"Cellname_final" %in% colnames(merge.obj[[]])

Source cell 48 (one-based); original execution count 55.

In [ ]:
# label transfer後のオブジェクトとして保存
saveRDS(
  merge.obj,
  file = "adipose_mergeobj_preProject_sketchHarmony_labelTransferred.rds"
)

# 保存できたか確認
file.info("adipose_mergeobj_preProject_sketchHarmony_labelTransferred.rds")[, c("size","mtime")]
